# 02 — Building the document processing pipeline

> **Applied Labs** · **Domain**: MM · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/05_document_processing/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb)

**What you will learn**:
- How to create synthetic document images with Pillow for testing
- Building a text-based document classifier
- Using the OpenAI Vision API for structured field extraction
- Implementing cross-field validation with arithmetic and format checks
- Confidence-based routing: auto-approve, review, or manual
- End-to-end pipeline orchestration across all 5 test documents

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "Pillow>=10.0.0" "pandas>=2.0.0" "numpy>=1.24.0"

import os, json, re, textwrap, base64, io
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from enum import Enum

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Creating synthetic test documents

We generate **5 synthetic documents** as images using Pillow. Each has known
ground-truth fields so we can evaluate extraction accuracy later.

| # | Type | Vendor/Source | Key fields |
|---|------|--------------|------------|
| 1 | Invoice | Acme Supplies | 3 line items, $648 total |
| 2 | Invoice | GlobalTech Ltd | 2 line items, $1,350 total |
| 3 | Invoice | FreshFoods Co | 4 line items, $432 total |
| 4 | Receipt | QuickMart | 3 items, $23.97 total |
| 5 | Purchase Order | TechCorp | 2 items, $5,400 total |

In [ ]:
# Create 5 synthetic document images with Pillow

@dataclass
class GroundTruth:
    """Ground truth for a synthetic document."""
    doc_type: str
    fields: Dict[str, Any]

def draw_text_document(
    lines: List[str],
    width: int = 800,
    height: int = 1000,
    font_size: int = 16,
    title_size: int = 24,
) -> Image.Image:
    """Render lines of text onto a white image, simulating a document."""
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)

    # Use default font (available everywhere)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
        title_font = ImageFont.truetype("arial.ttf", title_size)
    except (OSError, IOError):
        font = ImageFont.load_default()
        title_font = font

    y = 40
    for line in lines:
        if line.startswith("# "):
            draw.text((50, y), line[2:], fill="black", font=title_font)
            y += title_size + 16
            # Draw a line under the title
            draw.line([(50, y), (width - 50, y)], fill="gray", width=1)
            y += 10
        elif line.startswith("---"):
            draw.line([(50, y + 5), (width - 50, y + 5)], fill="gray", width=1)
            y += 15
        elif line.startswith("| "):
            # Table row — draw with fixed spacing
            cols = [c.strip() for c in line.split("|")[1:-1]]
            col_width = (width - 100) // max(len(cols), 1)
            for j, col in enumerate(cols):
                draw.text((60 + j * col_width, y), col, fill="black", font=font)
            y += font_size + 8
        else:
            draw.text((50, y), line, fill="black", font=font)
            y += font_size + 6

    # Draw border
    draw.rectangle([(20, 20), (width - 20, height - 20)], outline="gray", width=2)
    return img

# ── Document 1: Invoice — Acme Supplies ──
doc1_lines = [
    "# INVOICE",
    "",
    "Invoice Number: INV-2024-001",
    "Date: 2024-03-15",
    "Vendor: Acme Supplies Inc.",
    "Bill To: TechCorp LLC",
    "",
    "---",
    "| Item           | Qty | Unit Price | Total    |",
    "| Widget A       | 10  | $25.00     | $250.00  |",
    "| Widget B       | 5   | $50.00     | $250.00  |",
    "| Service Fee    | 1   | $100.00    | $100.00  |",
    "---",
    "",
    "Subtotal: $600.00",
    "Tax (8%): $48.00",
    "Total: $648.00",
    "",
    "Payment Terms: Net 30",
]

gt1 = GroundTruth("invoice", {
    "invoice_number": "INV-2024-001", "vendor_name": "Acme Supplies Inc.",
    "date": "2024-03-15",
    "line_items": [
        {"description": "Widget A", "qty": 10, "unit_price": 25.0, "total": 250.0},
        {"description": "Widget B", "qty": 5, "unit_price": 50.0, "total": 250.0},
        {"description": "Service Fee", "qty": 1, "unit_price": 100.0, "total": 100.0},
    ],
    "subtotal": 600.0, "tax_rate": 0.08, "tax": 48.0, "total": 648.0,
})

# ── Document 2: Invoice — GlobalTech ──
doc2_lines = [
    "# INVOICE",
    "",
    "Invoice No: GT-2024-0455",
    "Invoice Date: 2024-04-02",
    "From: GlobalTech Ltd",
    "To: Pinnacle Industries",
    "",
    "---",
    "| Description       | Quantity | Rate     | Amount     |",
    "| Cloud Hosting     | 1        | $850.00  | $850.00    |",
    "| API Calls (10K)   | 5        | $100.00  | $500.00    |",
    "---",
    "",
    "Subtotal: $1,350.00",
    "Tax (0%): $0.00",
    "Total Due: $1,350.00",
    "",
    "Payment: Wire Transfer — Net 15",
]

gt2 = GroundTruth("invoice", {
    "invoice_number": "GT-2024-0455", "vendor_name": "GlobalTech Ltd",
    "date": "2024-04-02",
    "line_items": [
        {"description": "Cloud Hosting", "qty": 1, "unit_price": 850.0, "total": 850.0},
        {"description": "API Calls (10K)", "qty": 5, "unit_price": 100.0, "total": 500.0},
    ],
    "subtotal": 1350.0, "tax_rate": 0.0, "tax": 0.0, "total": 1350.0,
})

# ── Document 3: Invoice — FreshFoods ──
doc3_lines = [
    "# INVOICE",
    "",
    "Inv #: FF-7721",
    "Date: 2024-03-28",
    "Supplier: FreshFoods Co.",
    "Customer: Campus Dining LLC",
    "",
    "---",
    "| Product         | Qty  | Price   | Line Total |",
    "| Apples (box)    | 12   | $15.00  | $180.00    |",
    "| Bread (loaf)    | 20   | $4.50   | $90.00     |",
    "| Milk (gal)      | 10   | $6.00   | $60.00     |",
    "| Eggs (dozen)    | 15   | $5.00   | $75.00     |",
    "---",
    "",
    "Subtotal: $405.00",
    "Sales Tax (6.67%): $27.00",
    "Grand Total: $432.00",
]

gt3 = GroundTruth("invoice", {
    "invoice_number": "FF-7721", "vendor_name": "FreshFoods Co.",
    "date": "2024-03-28",
    "line_items": [
        {"description": "Apples (box)", "qty": 12, "unit_price": 15.0, "total": 180.0},
        {"description": "Bread (loaf)", "qty": 20, "unit_price": 4.5, "total": 90.0},
        {"description": "Milk (gal)", "qty": 10, "unit_price": 6.0, "total": 60.0},
        {"description": "Eggs (dozen)", "qty": 15, "unit_price": 5.0, "total": 75.0},
    ],
    "subtotal": 405.0, "tax_rate": 0.0667, "tax": 27.0, "total": 432.0,
})

# ── Document 4: Receipt — QuickMart ──
doc4_lines = [
    "# RECEIPT",
    "",
    "QuickMart Store #42",
    "123 Main St, Springfield",
    "Date: 2024-03-20 14:32",
    "Cashier: Sarah M.",
    "",
    "---",
    "| Item            | Price   |",
    "| Coffee (large)  | $4.99   |",
    "| Sandwich        | $8.99   |",
    "| Water bottle    | $1.99   |",
    "---",
    "",
    "Subtotal: $15.97",
    "Tax (8%): $1.28",
    "Total: $17.25",
    "Payment: Visa *4242",
    "",
    "Thank you for shopping!",
]

gt4 = GroundTruth("receipt", {
    "store_name": "QuickMart Store #42", "date": "2024-03-20",
    "line_items": [
        {"description": "Coffee (large)", "total": 4.99},
        {"description": "Sandwich", "total": 8.99},
        {"description": "Water bottle", "total": 1.99},
    ],
    "subtotal": 15.97, "tax": 1.28, "total": 17.25,
    "payment_method": "Visa *4242",
})

# ── Document 5: Purchase Order — TechCorp ──
doc5_lines = [
    "# PURCHASE ORDER",
    "",
    "PO Number: PO-2024-0088",
    "Date: 2024-03-25",
    "Requester: Engineering Dept",
    "Vendor: ServerParts Direct",
    "Ship To: TechCorp HQ, Bldg C",
    "Delivery Date: 2024-04-10",
    "",
    "---",
    "| Item             | Qty | Unit Cost  | Total      |",
    "| SSD 1TB NVMe     | 20  | $120.00    | $2,400.00  |",
    "| RAM 32GB DDR5    | 20  | $150.00    | $3,000.00  |",
    "---",
    "",
    "Subtotal: $5,400.00",
    "Tax: $0.00",
    "Total: $5,400.00",
    "",
    "Approved By: J. Chen, VP Engineering",
]

gt5 = GroundTruth("purchase_order", {
    "po_number": "PO-2024-0088", "date": "2024-03-25",
    "requester": "Engineering Dept", "vendor": "ServerParts Direct",
    "line_items": [
        {"description": "SSD 1TB NVMe", "qty": 20, "unit_price": 120.0, "total": 2400.0},
        {"description": "RAM 32GB DDR5", "qty": 20, "unit_price": 150.0, "total": 3000.0},
    ],
    "subtotal": 5400.0, "tax": 0.0, "total": 5400.0,
})

# ── Generate all images ──
all_docs = [
    ("doc_01_invoice_acme", doc1_lines, gt1),
    ("doc_02_invoice_globaltech", doc2_lines, gt2),
    ("doc_03_invoice_freshfoods", doc3_lines, gt3),
    ("doc_04_receipt_quickmart", doc4_lines, gt4),
    ("doc_05_po_techcorp", doc5_lines, gt5),
]

document_images: Dict[str, Image.Image] = {}
ground_truths: Dict[str, GroundTruth] = {}

for name, lines, gt in all_docs:
    img = draw_text_document(lines)
    document_images[name] = img
    ground_truths[name] = gt

print("Synthetic document generation")
print("=" * 55)
for name, gt in ground_truths.items():
    img = document_images[name]
    print(f"  ✓ {name}: {gt.doc_type} ({img.size[0]}×{img.size[1]})")
    n_fields = len([k for k in gt.fields if k != "line_items"])
    n_items = len(gt.fields.get("line_items", []))
    print(f"    {n_fields} fields, {n_items} line items")

print(f"\n✓ {len(document_images)} documents created with ground truth")

## Section 2 — Document classifier

We build a keyword-based classifier that determines the document type from its
text content. This selects the extraction schema for the VLM prompt.

In [ ]:
# Document classifier — keyword matching on text content

CLASSIFICATION_KEYWORDS: Dict[str, List[str]] = {
    "invoice": ["invoice", "bill to", "amount due", "payment terms", "inv", "invoice number", "subtotal"],
    "receipt": ["receipt", "cashier", "change due", "thank you", "store", "payment method"],
    "purchase_order": ["purchase order", "po number", "requisition", "delivery date", "ship to", "approved by"],
    "form": ["application", "form", "please fill", "signature required", "applicant"],
    "report": ["report", "executive summary", "findings", "recommendations", "prepared by"],
}

def classify_document(text: str) -> Tuple[str, float]:
    """Classify document type from text using keyword matching."""
    text_lower = text.lower()
    scores: Dict[str, int] = {}
    for doc_type, keywords in CLASSIFICATION_KEYWORDS.items():
        scores[doc_type] = sum(1 for kw in keywords if kw in text_lower)
    best_type = max(scores, key=scores.get)
    total = sum(scores.values())
    confidence = scores[best_type] / max(total, 1)
    return best_type, round(confidence, 2)

# ── Test classifier on all 5 documents ──
print("Document classification results")
print("=" * 65)
correct = 0
for name, lines, gt in all_docs:
    text = "\n".join(lines)
    predicted, conf = classify_document(text)
    match = predicted == gt.doc_type
    if match:
        correct += 1
    icon = "✓" if match else "✗"
    print(f"  {icon} {name}")
    print(f"    Expected: {gt.doc_type:<18s}  Predicted: {predicted:<18s}  Conf: {conf:.2f}")

print(f"\nClassification accuracy: {correct}/{len(all_docs)} ({correct/len(all_docs)*100:.0f}%)")

## Section 3 — VLM extraction with OpenAI Vision API

We send each document image to **GPT-4o** with a structured extraction prompt.
The model returns JSON with all extracted fields and per-field confidence scores.

> **Note**: This cell requires a valid `OPENAI_API_KEY` environment variable.
> When running without an API key, we use pre-computed mock extractions for
> demonstration.

In [ ]:
# VLM extraction — OpenAI Vision API with fallback to mock data
from openai import OpenAI

def image_to_base64(img: Image.Image) -> str:
    """Convert a PIL Image to a base64 string."""
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

INVOICE_SCHEMA = {
    "invoice_number": "string",
    "vendor_name": "string",
    "date": "YYYY-MM-DD",
    "line_items": [{"description": "str", "qty": "int", "unit_price": "float", "total": "float"}],
    "subtotal": "number",
    "tax_rate": "number (decimal)",
    "tax": "number",
    "total": "number",
    "confidence": {"field_name": "0.0-1.0"},
}

RECEIPT_SCHEMA = {
    "store_name": "string",
    "date": "YYYY-MM-DD",
    "line_items": [{"description": "str", "total": "float"}],
    "subtotal": "number",
    "tax": "number",
    "total": "number",
    "payment_method": "string",
    "confidence": {"field_name": "0.0-1.0"},
}

PO_SCHEMA = {
    "po_number": "string",
    "date": "YYYY-MM-DD",
    "requester": "string",
    "vendor": "string",
    "line_items": [{"description": "str", "qty": "int", "unit_price": "float", "total": "float"}],
    "subtotal": "number",
    "tax": "number",
    "total": "number",
    "confidence": {"field_name": "0.0-1.0"},
}

def extract_with_vlm(
    img: Image.Image,
    doc_type: str,
    api_key: Optional[str] = None,
) -> Dict[str, Any]:
    """Extract structured fields from a document image using GPT-4o Vision."""
    schema = {"invoice": INVOICE_SCHEMA, "receipt": RECEIPT_SCHEMA,
              "purchase_order": PO_SCHEMA}.get(doc_type, INVOICE_SCHEMA)

    if not api_key:
        return {}  # caller handles fallback

    client = OpenAI(api_key=api_key)
    b64 = image_to_base64(img)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": (
                "You are a document extraction specialist. Extract all fields from the "
                "document image. Return ONLY valid JSON matching the schema. Include "
                "per-field confidence scores (0.0-1.0)."
            )},
            {"role": "user", "content": [
                {"type": "text", "text": (
                    f"Extract fields from this {doc_type}. "
                    f"Schema: {json.dumps(schema, indent=2)}"
                )},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{b64}", "detail": "high"
                }},
            ]},
        ],
        temperature=0.0,
        max_tokens=2000,
    )
    raw = response.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return json.loads(raw)

# ── Mock extractions (used when no API key) ──
MOCK_EXTRACTIONS: Dict[str, Dict[str, Any]] = {
    "doc_01_invoice_acme": {
        "invoice_number": "INV-2024-001", "vendor_name": "Acme Supplies Inc.",
        "date": "2024-03-15",
        "line_items": [
            {"description": "Widget A", "qty": 10, "unit_price": 25.0, "total": 250.0},
            {"description": "Widget B", "qty": 5, "unit_price": 50.0, "total": 250.0},
            {"description": "Service Fee", "qty": 1, "unit_price": 100.0, "total": 100.0},
        ],
        "subtotal": 600.0, "tax_rate": 0.08, "tax": 48.0, "total": 648.0,
        "confidence": {"invoice_number": 0.99, "vendor_name": 0.97, "date": 0.99,
                        "line_items": 0.95, "subtotal": 0.98, "tax": 0.96, "total": 0.99},
    },
    "doc_02_invoice_globaltech": {
        "invoice_number": "GT-2024-0455", "vendor_name": "GlobalTech Ltd",
        "date": "2024-04-02",
        "line_items": [
            {"description": "Cloud Hosting", "qty": 1, "unit_price": 850.0, "total": 850.0},
            {"description": "API Calls (10K)", "qty": 5, "unit_price": 100.0, "total": 500.0},
        ],
        "subtotal": 1350.0, "tax_rate": 0.0, "tax": 0.0, "total": 1350.0,
        "confidence": {"invoice_number": 0.98, "vendor_name": 0.96, "date": 0.99,
                        "line_items": 0.93, "subtotal": 0.97, "tax": 0.99, "total": 0.98},
    },
    "doc_03_invoice_freshfoods": {
        "invoice_number": "FF-7721", "vendor_name": "FreshFoods Co.",
        "date": "2024-03-28",
        "line_items": [
            {"description": "Apples (box)", "qty": 12, "unit_price": 15.0, "total": 180.0},
            {"description": "Bread (loaf)", "qty": 20, "unit_price": 4.5, "total": 90.0},
            {"description": "Milk (gal)", "qty": 10, "unit_price": 6.0, "total": 60.0},
            {"description": "Eggs (dozen)", "qty": 15, "unit_price": 5.0, "total": 75.0},
        ],
        "subtotal": 405.0, "tax_rate": 0.0667, "tax": 27.0, "total": 432.0,
        "confidence": {"invoice_number": 0.95, "vendor_name": 0.94, "date": 0.98,
                        "line_items": 0.91, "subtotal": 0.96, "tax": 0.88, "total": 0.97},
    },
    "doc_04_receipt_quickmart": {
        "store_name": "QuickMart Store #42", "date": "2024-03-20",
        "line_items": [
            {"description": "Coffee (large)", "total": 4.99},
            {"description": "Sandwich", "total": 8.99},
            {"description": "Water bottle", "total": 1.99},
        ],
        "subtotal": 15.97, "tax": 1.28, "total": 17.25,
        "payment_method": "Visa *4242",
        "confidence": {"store_name": 0.97, "date": 0.95, "line_items": 0.92,
                        "subtotal": 0.98, "tax": 0.94, "total": 0.99, "payment_method": 0.90},
    },
    "doc_05_po_techcorp": {
        "po_number": "PO-2024-0088", "date": "2024-03-25",
        "requester": "Engineering Dept", "vendor": "ServerParts Direct",
        "line_items": [
            {"description": "SSD 1TB NVMe", "qty": 20, "unit_price": 120.0, "total": 2400.0},
            {"description": "RAM 32GB DDR5", "qty": 20, "unit_price": 150.0, "total": 3000.0},
        ],
        "subtotal": 5400.0, "tax": 0.0, "total": 5400.0,
        "confidence": {"po_number": 0.99, "date": 0.98, "requester": 0.93,
                        "vendor": 0.96, "line_items": 0.94, "subtotal": 0.99, "total": 0.99},
    },
}

# ── Run extraction on all documents ──
api_key: Optional[str] = os.getenv("OPENAI_API_KEY")
extractions: Dict[str, Dict[str, Any]] = {}

print("VLM extraction results")
print("=" * 65)
for name in document_images:
    doc_type = ground_truths[name].doc_type
    if api_key:
        try:
            result = extract_with_vlm(document_images[name], doc_type, api_key)
            extractions[name] = result
            print(f"  ✓ {name}: extracted via GPT-4o")
        except Exception as e:
            extractions[name] = MOCK_EXTRACTIONS[name]
            print(f"  ⚠ {name}: API error, using mock — {e}")
    else:
        extractions[name] = MOCK_EXTRACTIONS[name]
        print(f"  ℹ {name}: using mock extraction (no API key)")

# ── Compare to ground truth ──
print("\n\nExtraction vs ground truth (key fields)")
print("=" * 65)
for name in document_images:
    gt = ground_truths[name].fields
    ex = extractions[name]
    print(f"\n  {name}:")
    # Compare scalar fields
    for key in ["invoice_number", "vendor_name", "store_name", "po_number", "date", "total"]:
        if key in gt:
            gt_val = gt[key]
            ex_val = ex.get(key, "MISSING")
            match = str(gt_val) == str(ex_val)
            icon = "✓" if match else "✗"
            print(f"    {icon} {key:20s}  GT: {str(gt_val):25s}  Extracted: {str(ex_val)}")

print(f"\n✓ {len(extractions)} documents processed")

## Section 4 — Cross-field validation

We validate each extraction against arithmetic rules, required fields, and
format constraints. Any failing check reduces confidence and may trigger
human review.

In [ ]:
# Cross-field validation engine

@dataclass
class ValidationIssue:
    """A validation issue found in an extraction."""
    rule_id: str
    severity: str  # error | warning
    field: str
    message: str

def validate_extraction(data: Dict[str, Any], doc_type: str) -> List[ValidationIssue]:
    """Validate extracted fields for arithmetic and format consistency."""
    issues: List[ValidationIssue] = []
    line_items = data.get("line_items", [])

    # ── Required fields ──
    required: Dict[str, List[str]] = {
        "invoice": ["invoice_number", "vendor_name", "date", "subtotal", "total"],
        "receipt": ["store_name", "date", "total"],
        "purchase_order": ["po_number", "vendor", "date", "total"],
    }
    for field_name in required.get(doc_type, []):
        val = data.get(field_name)
        if val is None or (isinstance(val, str) and val.strip() == ""):
            issues.append(ValidationIssue("REQ-001", "error", field_name,
                                          f"Required field '{field_name}' is missing"))

    # ── Line item arithmetic (invoices & POs) ──
    if doc_type in ("invoice", "purchase_order"):
        for i, item in enumerate(line_items):
            qty = item.get("qty", 0)
            price = item.get("unit_price", 0)
            total = item.get("total", 0)
            expected = qty * price
            if abs(expected - total) > 0.01:
                issues.append(ValidationIssue("ARITH-001", "error",
                    f"line_items[{i}].total",
                    f"Expected {expected:.2f} ({qty}×{price:.2f}), got {total:.2f}"))

    # ── Subtotal check ──
    if "subtotal" in data and line_items:
        if doc_type in ("invoice", "purchase_order"):
            expected_sub = sum(item.get("total", 0) for item in line_items)
        else:
            expected_sub = sum(item.get("total", 0) for item in line_items)
        actual_sub = data.get("subtotal", 0)
        if abs(expected_sub - actual_sub) > 0.01:
            issues.append(ValidationIssue("ARITH-002", "error", "subtotal",
                f"Expected {expected_sub:.2f}, got {actual_sub:.2f}"))

    # ── Tax check (invoices) ──
    if doc_type == "invoice" and "tax_rate" in data:
        tax_rate = data.get("tax_rate", 0)
        subtotal = data.get("subtotal", 0)
        expected_tax = round(subtotal * tax_rate, 2)
        actual_tax = data.get("tax", 0)
        if abs(expected_tax - actual_tax) > 0.02:
            issues.append(ValidationIssue("ARITH-003", "warning", "tax",
                f"Expected {expected_tax:.2f} ({subtotal:.2f}×{tax_rate}), got {actual_tax:.2f}"))

    # ── Grand total check ──
    subtotal = data.get("subtotal", 0)
    tax = data.get("tax", 0)
    expected_total = subtotal + tax
    actual_total = data.get("total", 0)
    if abs(expected_total - actual_total) > 0.01:
        issues.append(ValidationIssue("ARITH-004", "error", "total",
            f"Expected {expected_total:.2f} ({subtotal:.2f}+{tax:.2f}), got {actual_total:.2f}"))

    # ── Date format ──
    date_val = str(data.get("date", ""))
    if date_val and not re.match(r"^\d{4}-\d{2}-\d{2}$", date_val):
        issues.append(ValidationIssue("FMT-001", "warning", "date",
            f"Date '{date_val}' not in YYYY-MM-DD format"))

    return issues

# ── Validate all extractions ──
validation_results: Dict[str, List[ValidationIssue]] = {}

print("Validation results")
print("=" * 65)
for name in extractions:
    doc_type = ground_truths[name].doc_type
    issues = validate_extraction(extractions[name], doc_type)
    validation_results[name] = issues

    print(f"\n  {name} ({doc_type}):")
    if not issues:
        print("    ✓ All checks passed")
    for issue in issues:
        icon = "✗" if issue.severity == "error" else "⚠"
        print(f"    {icon} [{issue.rule_id}] {issue.field}: {issue.message}")

    n_errors = sum(1 for i in issues if i.severity == "error")
    n_warnings = sum(1 for i in issues if i.severity == "warning")
    print(f"    Summary: {n_errors} errors, {n_warnings} warnings")

total_issues = sum(len(v) for v in validation_results.values())
print(f"\n✓ Validated {len(validation_results)} documents — {total_issues} total issues found")

## Section 5 — Confidence-based routing

We compute a combined confidence score from VLM extraction confidence and
validation results, then route each document: auto-approve, quick review,
or manual entry.

In [ ]:
# Confidence scoring and routing

@dataclass
class RoutingDecision:
    """Final routing decision for a document."""
    doc_id: str
    doc_type: str
    extraction_confidence: float
    validation_score: float
    combined_confidence: float
    route: str
    reason: str

def compute_routing(
    doc_id: str,
    doc_type: str,
    extraction: Dict[str, Any],
    issues: List[ValidationIssue],
    total_rules: int = 8,
) -> RoutingDecision:
    """Compute confidence and routing for a processed document."""
    # Extraction confidence from VLM
    conf_dict = extraction.get("confidence", {})
    if conf_dict:
        ext_conf = sum(conf_dict.values()) / len(conf_dict)
    else:
        ext_conf = 0.80  # default if no confidence provided

    # Validation score
    n_errors = sum(1 for i in issues if i.severity == "error")
    n_warnings = sum(1 for i in issues if i.severity == "warning")
    val_score = max(0.0, 1.0 - (n_errors * 0.15) - (n_warnings * 0.05))

    # Combined
    combined = 0.6 * ext_conf + 0.4 * val_score

    # Route
    if combined >= 0.95:
        route, reason = "auto_approve", "High confidence — no review needed"
    elif combined >= 0.70:
        route, reason = "quick_review", "Medium confidence — flagged for quick review"
    else:
        route, reason = "manual_entry", "Low confidence — requires manual processing"

    return RoutingDecision(
        doc_id=doc_id, doc_type=doc_type,
        extraction_confidence=round(ext_conf, 3),
        validation_score=round(val_score, 3),
        combined_confidence=round(combined, 3),
        route=route, reason=reason,
    )

# ── Route all documents ──
routing_decisions: List[RoutingDecision] = []

print("Confidence routing decisions")
print("=" * 70)
for name in extractions:
    doc_type = ground_truths[name].doc_type
    decision = compute_routing(name, doc_type, extractions[name], validation_results[name])
    routing_decisions.append(decision)

    icon = {"auto_approve": "🟢", "quick_review": "🟡", "manual_entry": "🔴"}[decision.route]
    print(f"\n  {icon} {decision.doc_id}")
    print(f"     Type              : {decision.doc_type}")
    print(f"     Extract. conf     : {decision.extraction_confidence:.3f}")
    print(f"     Validation score  : {decision.validation_score:.3f}")
    print(f"     Combined          : {decision.combined_confidence:.3f}")
    print(f"     Route             : {decision.route}")
    print(f"     Reason            : {decision.reason}")

# ── Summary ──
route_counts = {}
for d in routing_decisions:
    route_counts[d.route] = route_counts.get(d.route, 0) + 1
print("\nRouting summary:")
for route, count in route_counts.items():
    print(f"  {route}: {count} documents")

## Section 6 — End-to-end pipeline

We orchestrate the full pipeline: classify → extract → validate → route.
Each document flows through all stages with a single function call.

In [ ]:
# End-to-end pipeline orchestration

@dataclass
class PipelineResult:
    """Complete result from the document processing pipeline."""
    doc_id: str
    classified_type: str
    classification_confidence: float
    extracted_fields: Dict[str, Any]
    validation_issues: List[ValidationIssue]
    routing: RoutingDecision

def process_document(
    doc_id: str,
    lines: List[str],
    img: Image.Image,
    api_key: Optional[str] = None,
) -> PipelineResult:
    """Process a single document through the full pipeline."""
    # Stage 1: Classify
    text = "\n".join(lines)
    doc_type, cls_conf = classify_document(text)

    # Stage 2: Extract (VLM or mock)
    if api_key:
        try:
            extracted = extract_with_vlm(img, doc_type, api_key)
        except Exception:
            extracted = MOCK_EXTRACTIONS.get(doc_id, {})
    else:
        extracted = MOCK_EXTRACTIONS.get(doc_id, {})

    # Stage 3: Validate
    issues = validate_extraction(extracted, doc_type)

    # Stage 4: Route
    routing = compute_routing(doc_id, doc_type, extracted, issues)

    return PipelineResult(
        doc_id=doc_id,
        classified_type=doc_type,
        classification_confidence=cls_conf,
        extracted_fields=extracted,
        validation_issues=issues,
        routing=routing,
    )

# ── Process all 5 documents ──
pipeline_results: List[PipelineResult] = []
api_key = os.getenv("OPENAI_API_KEY")

print("End-to-end pipeline results")
print("=" * 70)
for name, lines, gt in all_docs:
    result = process_document(name, lines, document_images[name], api_key)
    pipeline_results.append(result)

    icon = {"auto_approve": "🟢", "quick_review": "🟡", "manual_entry": "🔴"}[result.routing.route]
    print(f"\n{icon} {result.doc_id}")
    print(f"  Classification : {result.classified_type} (conf={result.classification_confidence:.2f})")
    print(f"  Extraction     : {len(result.extracted_fields)} fields")
    print(f"  Validation     : {len(result.validation_issues)} issues")
    print(f"  Routing        : {result.routing.route} (combined={result.routing.combined_confidence:.3f})")

# ── Final summary table ──
print("\n\nPipeline summary")
print("=" * 70)
rows = []
for r in pipeline_results:
    rows.append({
        "Document": r.doc_id.replace("doc_0", "Doc "),
        "Type": r.classified_type,
        "Fields": len(r.extracted_fields),
        "Issues": len(r.validation_issues),
        "Confidence": f"{r.routing.combined_confidence:.3f}",
        "Route": r.routing.route,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\n✓ {len(pipeline_results)} documents processed end-to-end")

## Exercises

1. **Add a new document type** — Create a synthetic "expense report" document
   (doc_06) with employee name, department, 3 expense items, and a total.
   Add it to the pipeline and verify it classifies correctly. You will need
   to add keywords for the new type.

2. **Improve the classifier** — Replace keyword matching with a TF-IDF + cosine
   similarity approach. Train on the 5 sample documents and test on a held-out
   sixth document. Does accuracy improve?

3. **Error injection** — Modify one mock extraction to contain a deliberate
   arithmetic error (e.g., wrong line item total). Verify the validator catches
   it and the routing changes to manual entry.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Synthetic documents with known ground truth enable controlled testing |
| 2 | Keyword classification is simple but effective for distinct document types |
| 3 | VLM extraction with structured prompts produces field-level JSON output |
| 4 | Cross-field arithmetic validation catches extraction errors automatically |
| 5 | Confidence routing balances cost vs accuracy by tiering review levels |
| 6 | End-to-end orchestration makes each stage independently testable |

## What's Next

In **03 — Evaluate**, we measure field-level accuracy, validation effectiveness,
confidence calibration, and per-document cost across the full test set.

---

## References

1. OpenAI, *GPT-4o Vision Capabilities* — <https://platform.openai.com/docs/guides/vision>
2. Pillow Documentation, *Image Drawing* — <https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html>
3. Reducto, *Document AI Benchmarks*, 2024 — <https://reducto.ai/>
4. AWS, *Textract Best Practices* — <https://docs.aws.amazon.com/textract/latest/dg/bestpractices.html>